# 1D Burgers' Equation with PINO

Train a Physics-Informed Neural Operator ([Li et al., 2021](https://arxiv.org/abs/2111.03794)) on
the viscous Burgers' equation. The loss combines the operator data loss with a penalty on the PDE
residual

$$R(u) = u_t + u\, u_x - \nu\, u_{xx}.$$

The residual term pushes predictions to satisfy the physics even where training data is scarce.

In [ ]:
# Install keras-sciml, then choose a backend: "tensorflow", "torch" or "jax"
!pip install keras-sciml
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

## Hyperparameters

`NU` is the viscosity and `T_END` the time the operator maps to.

In [ ]:
N_SAMPLES = 40
NX = 32
T_END = 0.5
NU = 0.05
MODES = 8
WIDTH = 24
DEPTH = 2
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.01

## Load the data

In [ ]:
import keras

from keras_sciml.datasets import BurgersDataset
from keras_sciml.losses import RelativeL2Loss
from keras_sciml.models import PINO

keras.utils.set_random_seed(42)

ds = BurgersDataset(mode="pseudospectral", n_samples=N_SAMPLES, nx=NX, t_end=T_END, nu=NU, seed=42)
u0, ut = ds.as_tuple()
dx = 1.0 / NX
print(f"Initial conditions: {u0.shape}, solutions: {ut.shape}")

## Physics residual

The spatial derivatives use central finite differences on the predicted $u(x, T)$. The time derivative
is the difference quotient $(u(x, T) - u_0(x)) / T$ between the input and the prediction.

In [ ]:
def burgers_residual(u, inputs):
    u_mid, u_prev, u_next = u[:, 1:-1, :], u[:, :-2, :], u[:, 2:, :]
    u_x = (u_next - u_prev) / (2.0 * dx)
    u_xx = (u_next - 2.0 * u_mid + u_prev) / (dx ** 2)
    u_t = (u_mid - inputs[:, 1:-1, :]) / T_END
    return u_t + u_mid * u_x - NU * u_xx

## Define the model

In [ ]:
model = PINO(
    modes=MODES,
    width=WIDTH,
    depth=DEPTH,
    out_channels=1,
    residual_fn=burgers_residual,
    pde_weight=0.01,
)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=RelativeL2Loss(),
    metrics=[RelativeL2Loss(name="rel_l2")],
)

## Train

In [ ]:
history = model.fit(
    u0,
    ut,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=2,
)
print(f"Final total loss: {history.history['loss'][-1]:.4f}")